"""
Implementasi paper:
"Multi-class brain tumor classification using residual network
and global average pooling" — Kumar et al. (2021)

Referensi: https://doi.org/10.1007/s11042-020-10335-4

Konfigurasi PERSIS seperti di paper:
- Model      : ResNet50 + GlobalAveragePooling + FCD(softmax)
- Epochs     : 10
- Batch Size : 2
- Optimizer  : SGDM (SGD + Momentum)
- LR         : 0.0001
- Loss       : Binary Cross-Entropy
- CV         : 5-Fold Cross Validation
- Pretrained : ImageNet
"""


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import json
import shutil
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ─────────────────────────────────────────────
# KONFIGURASI (sesuai paper Table 1)
# ─────────────────────────────────────────────
DATA_DIR      = "/content/drive/MyDrive/mri_otakv2/Training_Augmentedv1"
OUTPUT_DIR    = "/content/drive/MyDrive/mri_otakv2/Hasil_RNGAPv1"
IMG_SIZE      = (224, 224)
INPUT_SHAPE   = (224, 224, 3)
BATCH_SIZE    = 2
EPOCHS        = 10
LEARNING_RATE = 0.0001
MOMENTUM      = 0.9
N_FOLDS       = 5
RANDOM_SEED   = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
# ─────────────────────────────────────────────
# LOAD DATASET (path + label, tanpa augmentasi)
# ─────────────────────────────────────────────
def load_dataset_paths(data_dir):
    class_names = sorted([
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    ])
    print(f"Kelas ditemukan: {class_names}")

    image_paths, labels = [], []
    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(data_dir, cls)
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('.jpg')):
                image_paths.append(os.path.join(cls_dir, fname))
                labels.append(idx)

    image_paths = np.array(image_paths)
    labels      = np.array(labels)

    print(f"Total gambar: {len(image_paths)}")
    for idx, cls in enumerate(class_names):
        print(f"  {cls}: {np.sum(labels == idx)} gambar")

    return image_paths, labels, class_names

In [ ]:
# ─────────────────────────────────────────────
# BUILD MODEL RNGAP (sesuai paper Fig. 2)
#
# Arsitektur:
#   ResNet50 (ImageNet, include_top=False)
#       ↓
#   GlobalAveragePooling2D
#       ↓
#   Dense(num_classes, softmax)
# ─────────────────────────────────────────────
def build_rngap_model(num_classes):
    """
    RNGAP Model persis seperti di paper Kumar et al. 2021.
    ResNet50 → GlobalAveragePooling → Dense(softmax)
    Tidak ada Dense tambahan, tidak ada Dropout, tidak ada BatchNorm tambahan.
    """
    base_model = ResNet50(
        weights="imagenet",     # pretrained ImageNet (sesuai paper Section 3)
        include_top=False,      # hapus head ResNet50 asli (avg pool + FC 1000)
        input_shape=INPUT_SHAPE
    )

    # Semua layer dilatih (full fine-tuning, sesuai paper)
    base_model.trainable = True

    # Bangun model sesuai
    inputs  = tf.keras.Input(shape=(224, 224, 3))
    x       = base_model(inputs, training=True)
    x       = layers.GlobalAveragePooling2D()(x)      # GAP — sesuai paper
    outputs = layers.Dense(num_classes, activation="softmax")(x)  # FC + softmax

    model = models.Model(inputs, outputs)

    # SGDM optimizer (sesuai paper Section 4.2)
    optimizer = tf.keras.optimizers.SGD(
        learning_rate=LEARNING_RATE,
        momentum=MOMENTUM,
        nesterov=False
    )

    # Binary cross-entropy (sesuai paper Section 4.2)
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
# ─────────────────────────────────────────────
# PLOT HISTORY PER FOLD
# ─────────────────────────────────────────────
def plot_fold_history(training, fold_num, fold_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"RNGAP Model — Fold {fold_num}", fontsize=14, fontweight="bold")

    # Accuracy
    axes[0].plot(training.training["accuracy"],     label="Train Acc",  color="royalblue")
    axes[0].plot(training.training["val_accuracy"], label="Val Acc",    color="tomato")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Loss
    axes[1].plot(training.training["loss"],     label="Train Loss", color="royalblue")
    axes[1].plot(training.training["val_loss"], label="Val Loss",   color="tomato")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(fold_dir, f"fold_{fold_num}_history.png"), dpi=150)
    plt.close()



In [ ]:
# ─────────────────────────────────────────────
# PLOT SUMMARY SEMUA FOLD (sesuai Table 3 paper)
# ─────────────────────────────────────────────
def plot_fold_summary(fold_results, save_dir):
    folds      = [r["fold"]         for r in fold_results]
    val_accs   = [r["val_accuracy"] for r in fold_results]
    val_losses = [r["val_loss"]     for r in fold_results]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("RNGAP — 5-Fold Cross Validation Summary",
                 fontsize=13, fontweight="bold")

    axes[0].bar(folds, val_accs, color="steelblue", alpha=0.85, edgecolor="navy")
    axes[0].axhline(np.mean(val_accs), color="tomato", linestyle="--",
                    label=f"Mean = {np.mean(val_accs):.4f}")
    axes[0].set_title("Validation Accuracy per Fold")
    axes[0].set_xlabel("Fold")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_xticks(folds)
    axes[0].set_ylim(0, 1.1)
    axes[0].legend()
    axes[0].grid(axis="y", alpha=0.3)

    axes[1].bar(folds, val_losses, color="salmon", alpha=0.85, edgecolor="darkred")
    axes[1].axhline(np.mean(val_losses), color="navy", linestyle="--",
                    label=f"Mean = {np.mean(val_losses):.4f}")
    axes[1].set_title("Validation Loss per Fold")
    axes[1].set_xlabel("Fold")
    axes[1].set_ylabel("Loss")
    axes[1].set_xticks(folds)
    axes[1].legend()
    axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "fold_summary.png"), dpi=150)
    plt.close()
    print(f"Plot summary disimpan: {os.path.join(save_dir, 'fold_summary.png')}")


In [ ]:
# ─────────────────────────────────────────────
# MAIN — 5-FOLD CROSS VALIDATION
# ─────────────────────────────────────────────
def main():
    print("=" * 60)
    print("  ResNet50 + GAP | 5-Fold CV | ")
    print("=" * 60)

    # Load dataset
    image_paths, labels, class_names = load_dataset_paths(DATA_DIR)
    num_classes = len(class_names)
    print(f"\nJumlah kelas: {num_classes} → {class_names}")

    # Simpan class mapping
    class_mapping = {i: name for i, name in enumerate(class_names)}
    with open(os.path.join(OUTPUT_DIR, "class_mapping.json"), "w") as f:
        json.dump(class_mapping, f, indent=2)

    # One-hot encode labels untuk binary cross-entropy
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=num_classes)

    # Konfigurasi Cross Validation
    skf          = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    fold_results = []
    best_val_acc = 0.0
    best_fold    = -1

    for fold_num, (train_idx, val_idx) in enumerate(skf.split(image_paths, labels), start=1):
        print(f"\n{'─'*60}")
        print(f"  FOLD {fold_num} / {N_FOLDS}")
        print(f"  Train: {len(train_idx)} gambar | Val: {len(val_idx)} gambar")
        print(f"{'─'*60}")

        fold_dir = os.path.join(OUTPUT_DIR, f"fold_{fold_num}")
        os.makedirs(fold_dir, exist_ok=True)

        # Buat dataset — label one-hot untuk binary cross-entropy
        def preprocess(path, label):
            img = tf.io.read_file(path)
            img = tf.image.decode_jpeg(img, channels=3)
            img = tf.cast(img, tf.float32)
            img = tf.keras.applications.resnet50.preprocess_input(img)
            return img, label

        train_ds = tf.data.Dataset.from_tensor_slices(
            (image_paths[train_idx], labels_onehot[train_idx])
        ).shuffle(len(train_idx), seed=RANDOM_SEED)\
         .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)\
         .batch(BATCH_SIZE)\
         .prefetch(tf.data.AUTOTUNE)

        val_ds = tf.data.Dataset.from_tensor_slices(
            (image_paths[val_idx], labels_onehot[val_idx])
        ).map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)\
         .batch(BATCH_SIZE)\
         .prefetch(tf.data.AUTOTUNE)

        # Inisialisasi model RNGAP
        model = build_rngap_model(num_classes)

        if fold_num == 1:
            model.summary()

        best_ckpt = os.path.join(fold_dir, f"best_model_fold{fold_num}.keras")
        callbacks = [
            ModelCheckpoint(
                filepath=best_ckpt,
                monitor="val_accuracy",
                save_best_only=True,
                verbose=1
            ),
            CSVLogger(os.path.join(fold_dir, f"log_fold{fold_num}.csv"))
        ]

        # Training
        print(f"\n[Training] Fold {fold_num} — {EPOCHS} Epochs")
        training = model.fit(
            train_ds,
            epochs=10,
            validation_data=val_ds,
            callbacks=callbacks,
            verbose=1
        )

        plot_fold_history(training, fold_num, fold_dir)

        # Evaluasi dengan model terbaik
        best_model = tf.keras.models.load_model(best_ckpt)
        val_loss, val_acc = best_model.evaluate(val_ds, verbose=0)
        print(f"\n  Best Val Acc Fold {fold_num}: {val_acc:.4f} | Val Loss: {val_loss:.4f}")


        # Simpan report
        with open(os.path.join(fold_dir, f"report_fold{fold_num}.txt"), "w") as f:
            f.write(f"Fold {fold_num} — RNGAP50 Model")
            f.write("=" * 50 + "\n\n")
            f.write(f"Val Accuracy : {val_acc:.4f}\n")
            f.write(f"Val Loss     : {val_loss:.4f}\n\n")

        fold_results.append({
            "fold":         fold_num,
            "val_accuracy": float(val_acc),
            "val_loss":     float(val_loss),
            "model_path":   best_ckpt
        })

        # Simpan model terbaik
        if val_acc > best_val_acc:
            best_val_acc     = val_acc
            best_fold        = fold_num
            best_global_path = os.path.join(OUTPUT_DIR, "best_model_overall.keras")
            shutil.copy(best_ckpt, best_global_path)
            print(f"  ★ Model terbaik diperbarui → Fold {fold_num} (Val Acc: {val_acc:.4f})")

        tf.keras.backend.clear_session()

    # ─ - - - - - - -
    # RINGKASAN AKHIR
    # ---------------
    val_accs   = [r["val_accuracy"] for r in fold_results]
    val_losses = [r["val_loss"]     for r in fold_results]

    print("\n" + "=" * 60)
    print("  HASIL 5-FOLD CROSS VALIDATION")
    print("=" * 60)
    print(f"\n  {'Fold':<8} {'Val Accuracy':<16} {'Val Loss'}")
    print(f"  {'─'*40}")
    for r in fold_results:
        marker = " ★" if r["fold"] == best_fold else ""
        print(f"  Fold {r['fold']:<4} {r['val_accuracy']:.4f}{'':>10} {r['val_loss']:.4f}{marker}")

    print(f"\n  Mean Accuracy : {np.mean(val_accs):.4f} ± {np.std(val_accs):.4f}")
    print(f"  Mean Loss     : {np.mean(val_losses):.4f} ± {np.std(val_losses):.4f}")
    print(f"\n  ★ Model Terbaik : Fold {best_fold} (Val Acc = {best_val_acc:.4f})")
    print(f"  ★ Disimpan di   : {best_global_path}")

    # Simpan summary JSON
    summary = {
        "paper": "Kumar et al. 2021 — Multi-class brain tumor classification using RNGAP",
        "config": {
            "model":          "ResNet50 + GlobalAveragePooling + Dense(softmax)",
            "epochs":         EPOCHS,
            "batch_size":     BATCH_SIZE,
            "optimizer":      "SGDM",
            "learning_rate":  LEARNING_RATE,
            "momentum":       MOMENTUM,
            "loss":           "binary_crossentropy",
            "n_folds":        N_FOLDS,
            "augmentation":   False,
            "pretrained":     "ImageNet"
        },
        "fold_results":      fold_results,
        "mean_val_accuracy": float(np.mean(val_accs)),
        "std_val_accuracy":  float(np.std(val_accs)),
        "mean_val_loss":     float(np.mean(val_losses)),
        "best_fold":         best_fold,
        "best_val_accuracy": float(best_val_acc),
        "best_model_path":   best_global_path
    }
    with open(os.path.join(OUTPUT_DIR, "cv_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    plot_fold_summary(fold_results, OUTPUT_DIR)
    print("\n  Training selesai!")

In [ ]:
# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────
if __name__ == "__main__":
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        print(f"GPU ditemukan: {[g.name for g in gpus]}")
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    else:
        print("GPU tidak ditemukan, menggunakan CPU.")

    main()

GPU ditemukan: ['/physical_device:GPU:0']
  ResNet50 + GAP | 5-Fold CV | 
Kelas ditemukan: ['glioma', 'meningioma', 'pituitari']
Total gambar: 7924
  glioma: 2644 gambar
  meningioma: 2632 gambar
  pituitari: 2648 gambar

Jumlah kelas: 3 → ['glioma', 'meningioma', 'pituitari']

────────────────────────────────────────────────────────────
  FOLD 1 / 5
  Train: 6339 gambar | Val: 1585 gambar
────────────────────────────────────────────────────────────


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │         6,147 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,593,859 (90.00 MB)

 Trainable params: 23,540,739 (89.80 MB)

 Non-trainable params: 53,120 (207.50 KB)


[Training] Fold 1 — 10 Epochs
Epoch 1/10
3170/3170 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.6698 - loss: 0.4736
Epoch 1: val_accuracy improved from None to 0.90221, saving model to /content/drive/MyDrive/mri_otakv2/Hasil_RNGAPv1/fold_1/best_model_fold1.keras

Epoch 1: finished saving model to /content/drive/MyDrive/mri_otakv2/Hasil_RNGAPv1/fold_1/best_model_fold1.keras
3170/3170 ━━━━━━━━━━━━━━━━━━━━ 252s 68ms/step - accuracy: 0.7525 - loss: 0.3883 - val_accuracy: 0.9022 - val_loss: 0.1864
Epoch 2/10
3169/3170 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.8647 - loss: 0.2286
Epoch 2: val_accuracy improved from 0.90221 to 0.93186, saving model to /content/drive/MyDrive/mri_otakv2/Hasil_RNGAPv1/fold_1/best_model_fold1.keras

Epoch 2: finished saving model to /content/drive/MyDrive/mri_otakv2/Hasil_RNGAPv1/fold_1/best_model_fold1.keras
3170/3170 ━━━━━━━━━━━━━━━━━━━━ 147s 46ms/step - accuracy: 0.8839 - loss: 0.2014 - val_accuracy: 0.9319 - val_loss: 0.1180
Epoch 3/10
3169/3170 ━━━